### Project setup and problem statement

**Problem:**  
Predict whether a Titanic passenger survived (1) or not (0) using their profile and ticket information.

**Goal:**  
Build a reproducible ML pipeline that:
- Cleans and preprocesses the data.
- Engineers meaningful features.
- Trains and tunes a classifier.
- Evaluates performance with appropriate metrics.
- Saves the final pipeline for future use.

**Success criteria:**  
- Test ROC AUC ≥ 0.80 (or your own target).
- Clean, readable code with minimal leakage risk.
- Saved model file that can be loaded and used for predictions.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)
import joblib
import matplotlib.pyplot as plt

# Load data
notebook_path = Path.cwd()
repo_root = notebook_path
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

data_dir = repo_root / "data"
titanic_path = data_dir / "titanic.csv"

df = pd.read_csv(titanic_path)

# Standardize column names (adjust to your CSV)
df = df.rename(columns={
    "Survived": "survived",
    "Pclass": "pclass",
    "Age": "age",
    "SibSp": "sibsp",
    "Parch": "parch",
    "Fare": "fare",
    "Sex": "sex",
    "Embarked": "embarked"
})

df["class"] = df["pclass"].map({1: "First", 2: "Second", 3: "Third"})

print("Columns:", df.columns.tolist())
display(df.head(3))


Columns: ['PassengerId', 'survived', 'pclass', 'Name', 'sex', 'age', 'sibsp', 'parch', 'Ticket', 'fare', 'Cabin', 'embarked', 'class']


,PassengerId,survived,pclass,Name,sex,age,sibsp,parch,Ticket,fare,Cabin,embarked,class
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Third
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,First
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Third


### Basic cleaning and feature engineering

We perform minimal but meaningful steps:

- Drop columns that won't be used as features.
- Handle missing values in key columns.
- Create a few engineered features:
  - family_size
  - is_alone
  - age_bin (optional, if you want categorical age)


In [2]:
# Drop unnecessary columns
cols_to_drop = [c for c in ["deck", "embark_town", "alive", "who", "adult_male", "name", "ticket", "boat"] 
                if c in df.columns]
df = df.drop(columns=cols_to_drop)

# Basic imputation
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.dropna(subset=["fare"])

# Feature engineering
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"] = (df["family_size"] == 1).astype(int)

# Define target and features
target_col = "survived"
base_features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "class"]
engineered_features = ["family_size", "is_alone"]

feature_cols = base_features + engineered_features

X = df[feature_cols]
y = df[target_col]

print("Feature columns:", feature_cols)
print("X shape:", X.shape, "y shape:", y.shape)


Feature columns: ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'family_size', 'is_alone']
X shape: (891, 10) y shape: (891,)


### Train/test split

We split once and keep the test set untouched until final evaluation.
All preprocessing and tuning happens on the training set only.


In [3]:
# Identify numeric and categorical columns
numeric_features = ["age", "sibsp", "parch", "fare", "family_size"]
categorical_features = ["pclass", "sex", "embarked", "class", "is_alone"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)


Train: (712, 10) (712,)
Test: (179, 10) (179,)


### Preprocessing pipeline

We build a ColumnTransformer that:
- Imputes and scales numeric features.
- Imputes and one-hot encodes categorical features.

This preprocessor will be reused across models.


In [4]:
# Numeric transformer
numeric_transformer = Pipeline(steps=[
    ("num_imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical transformer
categorical_transformer = Pipeline(steps=[
    ("cat_imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)


### Model candidates

We compare two models:

1. Logistic Regression (linear, interpretable)
2. Random Forest (non-linear, often stronger)

Both are wrapped in pipelines with the same preprocessor.


In [5]:
# Logistic Regression pipeline
log_reg_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Random Forest pipeline
rf_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(random_state=42, n_jobs=-1))
])


### Model selection with cross-validation

We use 5-fold stratified CV to compare models on:
- ROC AUC (primary metric)
- Accuracy (secondary)

The better model will be tuned further.


In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["roc_auc", "accuracy"]

# Cross-validate Logistic Regression
cv_log = cross_validate(
    log_reg_pipe,
    X_train, y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

# Cross-validate Random Forest
cv_rf = cross_validate(
    rf_pipe,
    X_train, y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

# Summarize
def summarize(name, results):
    print(f"\n{name}:")
    for metric in scoring:
        key = f"test_{metric}"
        print(f"  {metric}: {results[key].mean():.3f} ± {results[key].std():.3f}")

summarize("Logistic Regression", cv_log)
summarize("Random Forest", cv_rf)



Logistic Regression:
  roc_auc: 0.856 ± 0.021
  accuracy: 0.796 ± 0.017

Random Forest:
  roc_auc: 0.863 ± 0.006
  accuracy: 0.802 ± 0.024


### Hyperparameter tuning

We tune the Random Forest (if it performed better) with GridSearchCV.

Parameters:
- n_estimators
- max_depth
- min_samples_leaf

Scoring: ROC AUC.


In [7]:
# Parameter grid for Random Forest
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 5]
}

grid_search = GridSearchCV(
    rf_pipe,
    param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

grid_search.fit(X_train, y_train)

print("Best CV ROC AUC:", round(grid_search.best_score_, 3))
print("Best params:", grid_search.best_params_)

best_model = grid_search.best_estimator_


Best CV ROC AUC: 0.878
Best params: {'model__max_depth': None, 'model__min_samples_leaf': 5, 'model__n_estimators': 200}


### Final evaluation on test set

We evaluate the best tuned model once on the held-out test set.

Metrics:
- ROC AUC (primary)
- Accuracy
- Confusion matrix
- Classification report


In [8]:
# Predictions
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# Metrics
test_roc_auc = roc_auc_score(y_test, y_proba)
test_acc = accuracy_score(y_test, y_pred)

print(f"Test ROC AUC: {test_roc_auc:.3f}")
print(f"Test Accuracy: {test_acc:.3f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3))


Test ROC AUC: 0.836
Test Accuracy: 0.799

Confusion matrix:
[[102   8]
 [ 28  41]]

Classification report:
              precision    recall  f1-score   support

           0      0.785     0.927     0.850       110
           1      0.837     0.594     0.695        69

    accuracy                          0.799       179
   macro avg      0.811     0.761     0.772       179
weighted avg      0.805     0.799     0.790       179



### Save the final pipeline

We save the entire pipeline (preprocessor + model) so it can be loaded later for predictions on new data.


In [10]:
from pathlib import Path

# Current working directory (usually notebooks/)
notebook_path = Path.cwd()

# Go up one level to repo root
repo_root = notebook_path.parent if notebook_path.name == "notebooks" else notebook_path

# Define models directory at repo root
models_dir = repo_root / "models"
models_dir.mkdir(exist_ok=True)

# Save path
model_path = models_dir / "titanic_survival_pipeline.joblib"
joblib.dump(best_model, model_path)

print(f"Model saved to: {model_path}")


Model saved to: /Users/tarun/tarun/genai/code/gen-ai-test/week01/models/titanic_survival_pipeline.joblib
